# PT-ROLLOUT-LIFECYCLE. Rollout and Policy Freshness | 轨迹与策略新鲜度

**难度：** Hard | **环境：** CPU-first | **标签：** `后训练`, `轨迹生命周期`, `策略新鲜度`

---

## 本节导读

在线后训练不能把生成结果直接送进下一次更新。每条轨迹都由确定的模型、策略版本和采样配置生成，还会经历执行、奖励、准入、消费和失效等状态；任一身份或状态缺失，都可能把错误数据混入参数更新。

轨迹依次经历 `生成 → 奖励 → 准入 → 预留 → 消费`。只有必需身份字段齐全、版本与行为偏离均在阈值内且尚未消费的轨迹才能绑定到一次更新；新策略产物只有在全部文件写入完成、且所有必需工作进程确认相同版本和哈希后才能激活。

**关键词：** 轨迹身份、策略版本、新鲜度、版本同步

![轨迹与策略版本生命周期](../docs/public/02_PyTorch_Algorithms/pt_rollout_lifecycle.svg)

## 前置阅读

**导语：** 按准备进入的在线更新分支选择前置页，再观察策略版本变化如何影响轨迹准入。

- PPO / GRPO 分支：[Part 02 · PT-REWARD-SIGNAL 奖励、价值与优势信号](./PT-REWARD-SIGNAL_Reward_Model_Value_and_Advantage.ipynb)
- Online DPO 分支：[Part 02 · 50 偏好数据与对齐评测](./50_Preference_Data_and_Evaluation.ipynb) → [Part 02 · 17 DPO 损失](./17_DPO_Loss_Tutorial.ipynb)

### Step 1：轨迹身份必须同时描述来源、生成条件和处理状态

一条轨迹首先要有唯一的 `rollout_id` 和 `prompt_id`，用于去重并关联原始任务；还要绑定策略、模型与分词器身份，避免同一版本号指向不同产物。采样参数或其哈希决定轨迹在什么生成条件下产生，奖励或验证器版本则说明分数来自哪套规则。

处理状态必须与身份分开记录：生成失败、工具超时、奖励未完成和已经消费是不同状态，不能统一改写成零奖励。可用于在线更新的正常状态链是 `generated → rewarded → reserved(update_id) → consumed`；预留操作把轨迹绑定到一次更新，更新提交后才能标记为已消费，更新失败则按恢复规则释放或隔离预留。

因此，去重不能只在一个 Python 列表内完成。学习器要同时检查持久状态与本批身份，并把“检查可用”和“预留给本次更新”作为一个不可分割的状态迁移；否则两个并发学习器都可能读到同一条 `rewarded` 轨迹并重复更新。

| 字段 | 回答的问题 | 不满足时的动作 |
| --- | --- | --- |
| `rollout_id / prompt_id` | 这是哪次采样、对应哪个任务 | 缺失则无法去重和追溯，直接拒绝 |
| `policy_version / artifact_hash` | 哪个策略产物生成了轨迹 | 版本或哈希不一致时隔离 |
| `tokenizer / sampling_hash` | 使用了什么分词和采样条件 | 与当前实验契约不一致时拒绝或分组 |
| `status / lifecycle_state` | 生成是否成功，以及轨迹处于已奖励、已预留还是已消费 | 失败轨迹进入失败分析；非 `rewarded` 状态不得再次预留 |
| `reward_status / reward_version` | 奖励是否完成、来自哪套规则 | 未完成则等待；版本不匹配时隔离 |
| `tokens / logprobs / action_mask` | 更新所需行为数据是否完整 | 与算法契约不符时拒绝 |

### Step 2：策略新鲜度是准入门槛，不是 on-policy 证明

令当前学习器策略版本为 `v`，轨迹生成版本为 `v_r`，版本滞后为 `lag = v - v_r`。`lag=0` 只说明版本一致；轨迹是否满足算法假设，还取决于采样分布、旧策略对数概率和是否被重复消费。一个可执行的偏离检查，是在动作掩码内比较当前策略与行为策略对已采样 Token 的对数概率，记录平均绝对差；该值超过实验阈值时，即使版本仍在允许范围内也要拒绝。

允许有限异步时，可以在算法明确支持且行为偏离受控的条件下接纳小于阈值的旧轨迹。未来版本、缺失版本、超过版本阈值、行为偏离过大或已经被预留/消费的轨迹必须隔离。版本差因此只是首道门槛，行为证据是第二道门槛；二者都不应被单独写成普适的 on-policy 证明。

| 状态 | 条件 | 处理 |
| --- | --- | --- |
| fresh | `lag == 0` | 继续检查采样契约、行为对数概率和消费状态 |
| tolerated | `0 < lag <= max_lag` | 继续检查动作位置的行为偏离，满足阈值后才能接纳 |
| stale | `lag > max_lag` | 拒绝或转入离线缓冲区 |
| future | `lag < 0` | 版本登记错误，立即拒绝 |

### Step 3：新策略只有完成产物确认和工作进程切换后才能激活

学习器产出新权重后，先进入准备阶段：保存不可变产物并记录哈希、父版本和回滚指针，再经过分发、加载与工作进程确认。工作进程上报的版本号和产物哈希必须一致。只有协调器在所有必需工作进程准备完成后原子切换活动策略指针，版本才进入提交阶段；只写出检查点或只有部分工作进程加载成功，都不能声明新版本已经激活。

激活失败时，协调器保持旧策略为活动版本，隔离准备阶段误生成的新轨迹，并保留重试或回滚状态。奖励服务超时、更新失败、产物未提交和权重分发失败属于不同阶段，必须分别记录，不能用一个笼统的失败标记覆盖。

![策略更新与工作进程同步门槛](../docs/public/02_PyTorch_Algorithms/pt_rollout_version_gate.svg)

### Step 4：实现轨迹准入、原子预留与策略激活

下面四个函数依次校验轨迹契约，联合版本滞后与动作位置对数概率差判断策略兼容性，把合格轨迹原子预留给一次更新，并在不可变产物与所有必需工作进程的版本和哈希一致后激活新策略。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1：校验轨迹契约<br>`validate_rollout` | 校验来源身份、生成条件、处理状态和动作证据 | 失败、奖励未完成、非 `rewarded` 或动作证据错位均不能进入更新 | 身份缺失、状态迁移错误与掩码错位 |
| TODO 2：判断策略兼容性<br>`assess_policy_compatibility` | 联合版本滞后与动作位置平均绝对对数概率差 | 未来/过期版本或行为偏离超限均拒绝 | 当前版本、容忍旧版本、过期版本与偏离超限 |
| TODO 3：预留更新批次<br>`reserve_update_batch` | 把合格轨迹绑定到唯一 `update_id`，同时保留拒绝原因 | 批内身份唯一；只从 `rewarded` 转为 `reserved`；不原地改写输入 | 混合批次、重复身份、预留状态与输入不变性 |
| TODO 4：激活策略产物<br>`activate_policy_version` | 检查已提交产物及工作进程的版本—哈希确认 | 未提交、缺失确认、版本或哈希不一致时不得激活 | 全部确认、未提交产物、缺失工作进程与错误哈希 |

In [ ]:
from typing import Any, Dict, List


In [ ]:
def validate_rollout(record):
    """校验轨迹身份、状态迁移和动作证据，返回 (是否可用, 原因)。"""
    # ==========================================
    # TODO 1：依次校验来源身份、生成条件、生命周期状态与动作证据。
    # required = ???  # 轨迹、策略产物、分词器与采样配置的必需身份字段。
    # ==========================================
    raise NotImplementedError("TODO 1：校验轨迹契约")


def assess_policy_compatibility(record, current_log_probs, current_version, max_lag, max_mean_abs_log_ratio):
    """联合版本滞后与动作位置行为偏离判断策略兼容性。"""
    # ==========================================
    # TODO 2：计算版本状态和动作位置平均绝对对数概率差。
    # lag = ???  # 当前学习器版本减轨迹生成版本。
    # mean_abs_log_ratio = ???  # 只统计 action_mask 为真的位置。
    # ==========================================
    raise NotImplementedError("TODO 2：判断策略兼容性")


def reserve_update_batch(records, current_log_probs_by_id, current_version, update_id, max_lag=0, max_mean_abs_log_ratio=0.2):
    """筛选轨迹并返回绑定 update_id 的预留副本，不原地改写输入。"""
    # ==========================================
    # TODO 3：组合契约、批内去重与策略兼容性，并执行 rewarded → reserved。
    # reserved = ???  # 合格轨迹的副本，写入 lifecycle_state 与 reserved_by。
    # rejected = ???  # 保留轨迹身份与首要拒绝原因。
    # ==========================================
    raise NotImplementedError("TODO 3：预留更新批次")


def activate_policy_version(artifact, worker_states, required_workers):
    """只有已提交产物与必需工作进程的版本—哈希一致时才激活。"""
    # ==========================================
    # TODO 4：检查产物提交状态以及工作进程的版本—哈希确认。
    # missing_workers = ???  # 尚未报告加载结果的必需工作进程。
    # mismatched_workers = ???  # 版本或产物哈希不一致的 worker。
    # ==========================================
    raise NotImplementedError("TODO 4：激活策略版本")

### 测试

测试覆盖轨迹契约、版本与行为偏离两项门槛、预留状态迁移、输入不变性，以及策略产物激活失败。

In [ ]:
def _record(version=3, status='completed', reward_status='completed', tokens=None, lifecycle_state='rewarded', rollout_id='r-1'):
    token_values = [1, 2] if tokens is None else tokens
    return {
        'rollout_id': rollout_id, 'prompt_id': 'p-1',
        'policy_version': version, 'policy_artifact_hash': f'policy-{version}',
        'tokenizer_hash': 'tok-v1', 'sampling_hash': 'sample-v1',
        'status': status, 'reward_status': reward_status, 'lifecycle_state': lifecycle_state,
        'tokens': token_values, 'action_mask': [True] * len(token_values),
        'old_log_probs': [-0.2] * len(token_values),
    }

def test_rollout_contract():
    """身份、生成结果、奖励状态与生命周期状态必须同时有效。"""
    assert validate_rollout(_record()) == (True, 'ok')
    assert validate_rollout(_record(status='failed'))[0] is False
    assert validate_rollout(_record(reward_status='pending'))[0] is False
    assert validate_rollout(_record(tokens=[])) == (False, 'empty_tokens')
    assert validate_rollout(_record(lifecycle_state='consumed')) == (False, 'not_rewarded')
    misaligned = _record(); misaligned['action_mask'] = [True]
    assert validate_rollout(misaligned) == (False, 'misaligned_action_evidence')

def test_policy_compatibility():
    """版本允许不代表行为偏离一定可接受。"""
    fresh = assess_policy_compatibility(_record(3), [-0.21, -0.19], 3, 1, 0.05)
    assert fresh['accepted'] is True and fresh['freshness'] == 'fresh'
    tolerated = assess_policy_compatibility(_record(2), [-0.22, -0.18], 3, 1, 0.05)
    assert tolerated['accepted'] is True and tolerated['freshness'] == 'tolerated'
    stale = assess_policy_compatibility(_record(1), [-0.2, -0.2], 3, 1, 0.05)
    assert stale['reason'] == 'stale_policy'
    drifted = assess_policy_compatibility(_record(3), [-0.8, -0.7], 3, 1, 0.05)
    assert drifted['reason'] == 'policy_drift'

def test_batch_reservation():
    """预留必须执行状态迁移、批内去重，并保持输入记录不变。"""
    records = [
        _record(3, rollout_id='r-0'), _record(2, rollout_id='r-1'),
        _record(1, rollout_id='r-2'), _record(3, status='failed', rollout_id='r-3'),
        _record(3, rollout_id='r-0'),
    ]
    current_log_probs = {record['rollout_id']: [-0.21, -0.19] for record in records}
    result = reserve_update_batch(records, current_log_probs, 3, 'update-7', max_lag=1, max_mean_abs_log_ratio=0.05)
    assert len(result['reserved']) == 2
    assert all(item['lifecycle_state'] == 'reserved' and item['reserved_by'] == 'update-7' for item in result['reserved'])
    assert records[0]['lifecycle_state'] == 'rewarded'
    assert {item['reason'] for item in result['rejected']} == {'stale_policy', 'generation_failed', 'duplicate_rollout'}

def test_version_activation():
    """缺失确认或错误版本不能激活新策略。"""
    artifact = {'version': 4, 'artifact_hash': 'policy-4', 'committed': True}
    workers = {
        'rollout-0': {'version': 4, 'artifact_hash': 'policy-4'},
        'rollout-1': {'version': 4, 'artifact_hash': 'policy-4'},
    }
    ready = activate_policy_version(artifact, workers, ['rollout-0', 'rollout-1'])
    assert ready == {'activated': True, 'active_version': 4, 'active_artifact_hash': 'policy-4', 'missing_workers': [], 'mismatched_workers': []}
    wrong = {'rollout-0': workers['rollout-0'], 'rollout-1': {'version': 4, 'artifact_hash': 'other'}}
    blocked = activate_policy_version(artifact, wrong, ['rollout-0', 'rollout-1'])
    assert blocked['activated'] is False and blocked['mismatched_workers'] == ['rollout-1']
    uncommitted = activate_policy_version(artifact | {'committed': False}, workers, ['rollout-0'])
    assert uncommitted['activated'] is False and uncommitted['reason'] == 'artifact_not_committed'

test_rollout_contract(); test_policy_compatibility(); test_batch_reservation(); test_version_activation()
print('✅ rollout 生命周期机制测试通过')

---

🛑 **STOP HERE** 🛑

> 请先完成题目区并运行测试，再查看参考实现。

---

## 参考代码与解析

In [ ]:
def validate_rollout(record):
    required = (
        'rollout_id', 'prompt_id', 'policy_version', 'policy_artifact_hash',
        'tokenizer_hash', 'sampling_hash',
    )
    if any(key not in record for key in required):
        return False, 'missing_identity'
    if record.get('status') != 'completed':
        return False, 'generation_failed'
    if record.get('reward_status') != 'completed':
        return False, 'reward_not_ready'
    if record.get('lifecycle_state') != 'rewarded':
        return False, 'not_rewarded'
    if not record.get('tokens'):
        return False, 'empty_tokens'
    token_count = len(record['tokens'])
    if len(record.get('action_mask', [])) != token_count or len(record.get('old_log_probs', [])) != token_count:
        return False, 'misaligned_action_evidence'
    return True, 'ok'

def assess_policy_compatibility(record, current_log_probs, current_version, max_lag, max_mean_abs_log_ratio):
    if max_lag < 0 or max_mean_abs_log_ratio < 0:
        raise ValueError('兼容性阈值必须非负')
    if len(current_log_probs) != len(record['old_log_probs']):
        raise ValueError('当前与行为策略对数概率必须同形')
    lag = current_version - record['policy_version']
    freshness = 'future' if lag < 0 else ('fresh' if lag == 0 else ('tolerated' if lag <= max_lag else 'stale'))
    active_differences = [
        abs(float(current) - float(old))
        for current, old, active in zip(current_log_probs, record['old_log_probs'], record['action_mask'])
        if active
    ]
    if not active_differences:
        return {'accepted': False, 'reason': 'empty_action_mask', 'freshness': freshness, 'mean_abs_log_ratio': None}
    mean_abs_log_ratio = sum(active_differences) / len(active_differences)
    if freshness in {'future', 'stale'}:
        return {'accepted': False, 'reason': f'{freshness}_policy', 'freshness': freshness, 'mean_abs_log_ratio': mean_abs_log_ratio}
    if mean_abs_log_ratio > max_mean_abs_log_ratio:
        return {'accepted': False, 'reason': 'policy_drift', 'freshness': freshness, 'mean_abs_log_ratio': mean_abs_log_ratio}
    return {'accepted': True, 'reason': 'ok', 'freshness': freshness, 'mean_abs_log_ratio': mean_abs_log_ratio}

def reserve_update_batch(records, current_log_probs_by_id, current_version, update_id, max_lag=0, max_mean_abs_log_ratio=0.2):
    if not update_id:
        raise ValueError('update_id 不能为空')
    reserved, rejected, seen_rollout_ids = [], [], set()
    for record in records:
        rollout_id = record.get('rollout_id')
        if rollout_id in seen_rollout_ids:
            rejected.append({'record': record, 'reason': 'duplicate_rollout'})
            continue
        seen_rollout_ids.add(rollout_id)
        valid, reason = validate_rollout(record)
        if not valid:
            rejected.append({'record': record, 'reason': reason})
            continue
        if rollout_id not in current_log_probs_by_id:
            rejected.append({'record': record, 'reason': 'missing_current_log_probs'})
            continue
        compatibility = assess_policy_compatibility(
            record, current_log_probs_by_id[rollout_id], current_version, max_lag, max_mean_abs_log_ratio
        )
        if not compatibility['accepted']:
            rejected.append({'record': record, 'reason': compatibility['reason']})
            continue
        reserved.append(record | {
            'lifecycle_state': 'reserved', 'reserved_by': update_id,
            'freshness': compatibility['freshness'],
            'mean_abs_log_ratio': compatibility['mean_abs_log_ratio'],
        })
    return {'update_id': update_id, 'reserved': reserved, 'rejected': rejected}

def activate_policy_version(artifact, worker_states, required_workers):
    # TODO 4：激活策略产物。
    if not artifact.get('committed', False):
        return {'activated': False, 'reason': 'artifact_not_committed', 'active_version': None,
                'active_artifact_hash': None, 'missing_workers': [], 'mismatched_workers': []}
    artifact_version = artifact['version']
    artifact_hash = artifact['artifact_hash']
    missing_workers = sorted(worker for worker in required_workers if worker not in worker_states)
    mismatched_workers = sorted(
        worker for worker in required_workers if worker in worker_states
        and (worker_states[worker].get('version') != artifact_version
             or worker_states[worker].get('artifact_hash') != artifact_hash)
    )
    activated = not missing_workers and not mismatched_workers
    return {'activated': activated, 'active_version': artifact_version if activated else None,
            'active_artifact_hash': artifact_hash if activated else None,
            'missing_workers': missing_workers, 'mismatched_workers': mismatched_workers}

### 解析

#### TODO 1：校验轨迹契约

- **机制：** 联合检查轨迹身份、策略与采样身份、奖励状态、生命周期状态和动作位置证据。
- **不变量：** 只有处于 `rewarded` 状态且动作掩码与行为策略对数概率对齐的轨迹可以进入预留。
- **测试证据：** 测试覆盖生成失败、奖励未完成、空轨迹、错误状态和动作证据错位。

#### TODO 2：判断策略兼容性

- **机制：** 先按策略版本差筛除未来或过期轨迹，再在动作位置计算当前策略与行为策略的平均绝对对数概率差。
- **不变量：** 版本差合格不代表行为仍兼容；任一门槛超限都必须拒绝轨迹。
- **测试证据：** 测试分别覆盖新鲜、可容忍滞后、过期和行为偏离四种情况。

#### TODO 3：预留更新批次

- **机制：** 对合格轨迹执行检查—预留状态迁移，将副本从 `rewarded` 改为 `reserved` 并绑定 `update_id`。
- **不变量：** 同一 `rollout_id` 不能在批内重复预留，输入记录不可原地修改，每条拒绝记录必须保留原因。
- **测试证据：** 测试核对重复身份拒绝、状态迁移、输入不变性和拒绝原因集合。

#### TODO 4：激活策略产物

- **机制：** 新策略产物先完成提交，再由所有必需工作进程确认相同版本与不可变哈希，最后切换活动版本。
- **不变量：** 未提交产物、缺失确认或版本—哈希不一致时都不能激活新策略。
- **测试证据：** 测试覆盖成功激活、哈希不一致和产物未提交三条路径。

## 相关阅读

- [Part 02 · PT-RL-SYSTEM 采样、训练与奖励系统](./PT-RL-SYSTEM_Rollout_Training_and_Reward_System.ipynb)
- [Part 02 · 51 Online DPO](./51_Online_DPO.ipynb)
- [TRL 文档](https://huggingface.co/docs/trl/)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)